# Delivery ETA Prediction — EDA & Modeling
Quick-commerce scenario (Zepto-style). This notebook is for exploration only.
Production code lives in `src/`.

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from data_processing import generate_dataset, clean
from feature_engineering import build_features, correlation_report

sns.set_theme(style='whitegrid')
raw = clean(generate_dataset(5000))
raw.head()

In [ ]:
raw.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
raw['eta_minutes'].hist(bins=50, ax=axes[0], color='steelblue'); axes[0].set_title('ETA Distribution')
raw['distance_km'].hist(bins=50, ax=axes[1], color='coral');    axes[1].set_title('Distance Distribution')
raw.groupby('hour')['eta_minutes'].mean().plot(ax=axes[2], marker='o'); axes[2].set_title('Avg ETA by Hour')
plt.tight_layout(); plt.show()

In [ ]:
eng = build_features(raw)
corr = correlation_report(eng)
print(corr)

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(eng.corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Feature Correlation Matrix')
plt.tight_layout(); plt.show()

In [ ]:
# Quick model comparison
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
import xgboost as xgb

feat_cols = [c for c in eng.columns if c != 'eta_minutes']
X, y = eng[feat_cols].values, eng['eta_minutes'].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

for name, m in [('Ridge', Ridge()), ('RF', RandomForestRegressor(100, random_state=42)), ('XGB', xgb.XGBRegressor(verbosity=0))]:
    m.fit(X_tr, y_tr)
    mae = mean_absolute_error(y_te, m.predict(X_te))
    print(f'{name:6s}  MAE={mae:.3f} min')